#### Предсказываем цену ноутбука

В папке лежит датасет laptop_prices, в котором содержатся сведения о ноутбуках, их характеристики и цены. Обучите модель линейной регрессии, которая будет предсказывать цену ноутбука по его характеристикам. 

Придется хорошенько поработать с характеристиками: это *творческая* часть задания. Во-первых, надо привести их в машиночитаемый вид, а во-вторых, можно посмотреть, как они коррелируют друг с другом и не нужно ли кого-то из них дропнуть или наоборот. Не советую бездумно использовать OHE: некоторые признаки явно можно закодировать куда более оптимальным способом. 

*Примечание*: без работы над фичами за все дз - **0 баллов**. 

In [1]:
import pandas as pd

In [55]:
raw = pd.read_csv('laptop_prices.csv')
raw.head()

,Brand,Processor,RAM (GB),Storage,GPU,Screen Size (inch),Resolution,Battery Life (hours),Weight (kg),Operating System,Price ($)
0,Apple,AMD Ryzen 3,64,512GB SSD,Nvidia GTX 1650,17.3,2560x1440,8.9,1.42,FreeDOS,3997.07
1,Razer,AMD Ryzen 7,4,1TB SSD,Nvidia RTX 3080,14.0,1366x768,9.4,2.57,Linux,1355.78
2,Asus,Intel i5,32,2TB SSD,Nvidia RTX 3060,13.3,3840x2160,8.5,1.74,FreeDOS,2673.07
3,Lenovo,Intel i5,4,256GB SSD,Nvidia RTX 3080,13.3,1366x768,10.5,3.10,Windows,751.17
4,Razer,Intel i3,4,256GB SSD,AMD Radeon RX 6600,16.0,3840x2160,5.7,3.38,Linux,2059.83


In [56]:
raw['Resolution'].unique()

array(['2560x1440', '1366x768', '3840x2160', '1920x1080'], dtype=object)

In [ ]:
pd.pivot_table(raw, index='RAM (GB)', columns='Resolution', aggfunc='size')
# корреляции между признаками я не нашла

Resolution,1366x768,1920x1080,2560x1440,3840x2160
RAM (GB),,,,
4,586,577,602,596
8,544,591,564,597
16,631,553,559,618
32,588,602,619,601
64,583,607,585,565


In [ ]:
from matplotlib import pylab as plt

for c in raw.columns:
    if c != 'charges':
        print(c)
        plt.figure(figsize=(12,6))
        plt.scatter(raw[c], raw['Price ($)'])
        plt.show()

In [57]:
data1 = raw.drop(['Brand', 'Storage', 'GPU'], axis = 1)
# 'Screen Size (inch)', 'Battery Life (hours)', 'Weight (kg)'
# можно было еще их дропнуть, потому что корреляции нет, но раз уж числовые, пусть будут

In [58]:
data1 = pd.get_dummies(data1, 'Processor', 'Resolution', 'Operating System', drop_first=True) 

In [59]:
from sklearn.model_selection import train_test_split
X = data1.drop('Price ($)', axis = 1)
y = data1['Price ($)']
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=42)

In [60]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

model = LinearRegression()

model.fit(Xtrain, ytrain) 
pred_train = model.predict(Xtrain)
pred_test = model.predict(Xtest)
print(mean_squared_error(pred_test, ytest) ** 0.5, mean_squared_error(pred_train, ytrain) ** 0.5)
# модель не переобучена (ну разве что чут чут)

662.3993836596535 645.9550431358521


In [63]:
# попробуем дропнуть и числовые
data2 = raw.drop(['Brand', 'Storage', 'GPU', 'Screen Size (inch)', 'Battery Life (hours)', 'Weight (kg)'], axis = 1)

In [64]:
data2 = pd.get_dummies(data2, 'Processor', 'Resolution', 'Operating System', drop_first=True) 
X = data2.drop('Price ($)', axis = 1)
y = data2['Price ($)']
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=42)

model.fit(Xtrain, ytrain) 
pred_train = model.predict(Xtrain)
pred_test = model.predict(Xtest)
print(mean_squared_error(pred_test, ytest) ** 0.5, mean_squared_error(pred_train, ytrain) ** 0.5)
# лучше не стало

668.8990099883265 651.9177172111254
